# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

In [26]:
import re
import Bio.SeqIO
import Bio.Seq

#read sequences from fastq file and store as a list of Seq objects
seqreads = list(Bio.SeqIO.parse('barcodes_R1.fastq', 'fastq'))
seqreads_Seq = []
for seqrecord in seqreads:
    seqreads_Seq.append(seqrecord.seq)  

#identify barcode and gene with knownn upstream sequence
def read_barcode(seqread, bclen, upstream='AGGCGGCCGC'):
    rseq = str(seqread.reverse_complement()) #get reverse complement of sequence

    #indentify NA and HA reads using the unique barcodes 
    HA_pat = re.compile('CCGGATTTGC' + '.*?' + upstream + '(?P<barcode>[A-Z]{' + str(bclen) + '})')
    NA_pat = re.compile('CACGATAGAT' + '.*?' + upstream + '(?P<barcode>[A-Z]{' + str(bclen) + '})')  

    #find reads that match for HA barcode and assign identity as HA
    HA_match = HA_pat.search(rseq)
    if HA_match:
        return HA_match.group('barcode'), 'HA'

    #if they did match with HA, find reads that match with NA and assign to NA 
    NA_match = NA_pat.search(rseq)
    if NA_match:
        return NA_match.group('barcode'), 'NA'
        
    #if there is no macth for either barcode return none
    return None, None

In [27]:
#initilaize individual dictionaries for barcode counts for each gene 
HA_barcode_counts = {}
NA_barcode_counts = {}

#initialize invalid counter for individual genes
HA_invalid_count = 0
NA_invalid_count = 0

#loop through all reads and assign each HA or NA barcode counts to collect total number of each
for seq in seqreads_Seq:
    barcode, gene = read_barcode(seq, bclen=16)

    if gene == 'HA':
        if barcode in HA_barcode_counts:
            HA_barcode_counts[barcode] = HA_barcode_counts[barcode] + 1
        else:
            HA_barcode_counts[barcode] = 1

    elif gene == 'NA':
        if barcode in NA_barcode_counts:
            NA_barcode_counts[barcode] = NA_barcode_counts[barcode] + 1
        else:
            NA_barcode_counts[barcode] = 1

    else:
        # no valid barcode found for either HA or NA
        HA_invalid_count += 1
        NA_invalid_count += 1

1. How many reads map to HA, and how many reads map to NA?

In [28]:
#Read total number of HA and NA reads based on number of barcode counts for each
print('total HA reads:', sum(HA_barcode_counts.values()))
print('total NA reads:', sum(NA_barcode_counts.values()))

total HA reads: 5327
total NA reads: 3938


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [29]:
print('num invalid HA:', HA_invalid_count)
print('num invalid NA:', NA_invalid_count)
#Only able to identify the total number of of sequences without a valid barcode. Can not determine which are HA and which are NA if there is no valid barcode.
#The total number of invalid barcodes is 735

num invalid HA: 735
num invalid NA: 735


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [18]:
# find top HA barcode
top_HA_barcode, top_HA_count = '', 0
for barcode, count in HA_barcode_counts.items():
    if count > top_HA_count:
        top_HA_count = count
        top_HA_barcode = barcode
print(f'most common HA barcode: {top_HA_barcode}  count: {top_HA_count}')

most common HA barcode: CCCGACCCGACATTAA  count: 155


In [19]:
top_NA_barcode, top_NA_count = '', 0
for barcode, count in NA_barcode_counts.items():
    if count > top_NA_count:
        top_NA_count = count
        top_NA_barcode = barcode
print(f'most common NA barcode: {top_NA_barcode}  count: {top_NA_count}')

most common NA barcode: ACCAGTTCTCCCCGGG  count: 152
